# 01. Open Images 메타데이터 다운로드

Open Images V7 (validation + test 세트)에서 대상 클래스의
이미지 ID, bbox 좌표, 다운로드 URL을 수집합니다.

**출력**: `data/metadata/manifest.csv`

**실행 방법**: 셀을 순서대로 `Shift+Enter`로 실행하세요.

**다운로드 파일 크기 예상**
| 파일 | 크기 |
|------|------|
| class_descriptions.csv | ~1 MB |
| validation_bbox.csv | ~25 MB |
| test_bbox.csv | ~75 MB |
| validation_images.csv | ~5 MB |
| test_images.csv | ~15 MB |

In [1]:
import os
import sys
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# 노트북 위치에 상관없이 프로젝트 루트를 기준으로 실행
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config

for d in [config.METADATA_DIR, config.RAW_DIR, config.PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'프로젝트 루트 : {project_root}')
print(f'대상 클래스   : {config.OI_TARGET_CLASSES}')
print(f'클래스당 최대 : {config.MAX_SAMPLES_PER_CLASS}개')

프로젝트 루트 : C:\Users\SSAFY\Desktop\miribom
대상 클래스   : ['Washing machine', 'Refrigerator']
클래스당 최대 : 100개


In [2]:
def download_file(url, save_path, desc=''):
    """이미 존재하면 스킵, 없으면 스트리밍 다운로드."""
    if os.path.exists(save_path):
        print(f'[skip] {os.path.basename(save_path)}')
        return
    r = requests.get(url, stream=True, timeout=config.DOWNLOAD_TIMEOUT)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    label = desc or os.path.basename(save_path)
    with open(save_path, 'wb') as f, tqdm(total=total, unit='iB', unit_scale=True, desc=label) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    print(f'[완료] {save_path}')

In [3]:
# ── Step 1: 클래스 설명 CSV → 대상 클래스의 LabelName 코드 추출 ─────────────────
desc_path = os.path.join(config.METADATA_DIR, 'class_descriptions.csv')
download_file(config.OI_URLS['class_descriptions'], desc_path)

class_desc = pd.read_csv(desc_path, header=None, names=['LabelName', 'ClassName'])
target_rows = class_desc[class_desc['ClassName'].isin(config.OI_TARGET_CLASSES)]
label_to_class = dict(zip(target_rows['LabelName'], target_rows['ClassName']))

print('\n찾은 레이블 코드:')
print(target_rows.to_string(index=False))

missing = set(config.OI_TARGET_CLASSES) - set(target_rows['ClassName'])
if missing:
    print(f'[경고] config.OI_TARGET_CLASSES에 없는 클래스명: {missing}')
    print('  → oidv6-class-descriptions.csv 에서 정확한 영문 표기를 확인하세요.')

class_descriptions.csv:   0%|          | 0.00/459k [00:00<?, ?iB/s]

[완료] data\metadata\class_descriptions.csv

찾은 레이블 코드:
LabelName       ClassName
/m/0174k2 Washing machine
/m/040b_t    Refrigerator


In [4]:
# ── Step 2: Validation bbox CSV 다운로드 + 필터링 ───────────────────────────────
val_bbox_path = os.path.join(config.METADATA_DIR, 'validation_bbox.csv')
download_file(config.OI_URLS['validation_bbox'], val_bbox_path)

val_bbox = pd.read_csv(val_bbox_path)
val_filtered = val_bbox[val_bbox['LabelName'].isin(label_to_class)].copy()
val_filtered['ClassName'] = val_filtered['LabelName'].map(label_to_class)
val_filtered['Subset'] = 'validation'

print('Validation 필터 결과:')
print(val_filtered['ClassName'].value_counts())

validation_bbox.csv:   0%|          | 0.00/25.1M [00:00<?, ?iB/s]

[완료] data\metadata\validation_bbox.csv


Validation 필터 결과:
ClassName
Washing machine    40
Refrigerator       26
Name: count, dtype: int64


In [5]:
# ── Step 3: Test bbox CSV 다운로드 + 필터링 ──────────────────────────────────────
test_bbox_path = os.path.join(config.METADATA_DIR, 'test_bbox.csv')
download_file(config.OI_URLS['test_bbox'], test_bbox_path)

test_bbox = pd.read_csv(test_bbox_path)
test_filtered = test_bbox[test_bbox['LabelName'].isin(label_to_class)].copy()
test_filtered['ClassName'] = test_filtered['LabelName'].map(label_to_class)
test_filtered['Subset'] = 'test'

print('Test 필터 결과:')
print(test_filtered['ClassName'].value_counts())

test_bbox.csv:   0%|          | 0.00/77.5M [00:00<?, ?iB/s]

[완료] data\metadata\test_bbox.csv


Test 필터 결과:
ClassName
Refrigerator       125
Washing machine    123
Name: count, dtype: int64


In [6]:
# ── Step 4: 합치기 + 클래스별 MAX_SAMPLES 샘플링 ────────────────────────────────
# 동일 이미지 내 중복 bbox는 모두 유지 (이미지 단위 dedup 없음 → 더 다양한 crop 확보)
combined = pd.concat([val_filtered, test_filtered], ignore_index=True)

sampled_parts = []
for cls, group in combined.groupby('ClassName'):
    n = min(len(group), config.MAX_SAMPLES_PER_CLASS)
    sampled_parts.append(group.sample(n=n, random_state=42))
sampled = pd.concat(sampled_parts, ignore_index=True)

print('샘플링 결과 (bbox 단위):')
print(sampled['ClassName'].value_counts())

샘플링 결과 (bbox 단위):
ClassName
Refrigerator       100
Washing machine    100
Name: count, dtype: int64


In [7]:
# ── Step 5: 이미지 목록 CSV 다운로드 → OriginalURL 조인 ─────────────────────────
val_img_path  = os.path.join(config.METADATA_DIR, 'validation_images.csv')
test_img_path = os.path.join(config.METADATA_DIR, 'test_images.csv')
download_file(config.OI_URLS['validation_images'], val_img_path)
download_file(config.OI_URLS['test_images'], test_img_path)

val_imgs  = pd.read_csv(val_img_path,  usecols=['ImageID', 'OriginalURL'])
test_imgs = pd.read_csv(test_img_path, usecols=['ImageID', 'OriginalURL'])
all_imgs  = pd.concat([val_imgs, test_imgs], ignore_index=True)

manifest = sampled.merge(all_imgs, on='ImageID', how='left')

missing_url = manifest['OriginalURL'].isna().sum()
if missing_url:
    print(f'[경고] URL 없는 행 {missing_url}개 → 제외')
manifest = manifest.dropna(subset=['OriginalURL'])

print(f'최종 manifest 행 수: {len(manifest)}')

validation_images.csv:   0%|          | 0.00/15.2M [00:00<?, ?iB/s]

[완료] data\metadata\validation_images.csv


test_images.csv:   0%|          | 0.00/45.2M [00:00<?, ?iB/s]

[완료] data\metadata\test_images.csv


최종 manifest 행 수: 200


In [8]:
# ── Step 6: manifest.csv 저장 ──────────────────────────────────────────────────
cols = ['ImageID', 'ClassName', 'LabelName', 'XMin', 'XMax', 'YMin', 'YMax', 'Subset', 'OriginalURL']
manifest[cols].to_csv(config.MANIFEST_PATH, index=False)

print(f'manifest 저장 완료: {config.MANIFEST_PATH}')
print(f'총 bbox 수: {len(manifest)}')
print()
print(manifest['ClassName'].value_counts())
print()
print('다음 단계: 02_download_images.ipynb 실행')

manifest 저장 완료: data\metadata\manifest.csv
총 bbox 수: 200

ClassName
Refrigerator       100
Washing machine    100
Name: count, dtype: int64

다음 단계: 02_download_images.ipynb 실행
